# Hyperliquid

Download data from hyperliquid (https://hyperfoundation.org/). Hyperliquid is a DEX.

In [ ]:
#| default_exp hyperliquid

In [ ]:
#|hide
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#| export
from nbdev.showdoc import *
import json
from typing import List, Dict, Tuple, Optional, Union, Any, Callable
from hyperliquid.utils import constants
import os
import csv

import eth_account
from eth_account.signers.local import LocalAccount
from hyperliquid.exchange import Exchange
from hyperliquid.info import Info
import pandas as pd
from datetime import datetime
import numpy as np

In [ ]:
#| export
def setup(base_url=None, skip_ws=False, perp_dexs=None,config='../config_hyperliquid.json'):
    # This function is copied from hyperliquid-python-sdk/examples/example_utils.py
    # for setting up the environment in our script.
    # config_path = os.path.join(os.path.dirname(__file__), "config.json")
    config_path = config
    with open(config_path) as f:
        config = json.load(f)
    account: LocalAccount = eth_account.Account.from_key(config["secret_key"])
    address = config["account_address"]
    if address == "":
        address = account.address
    print("Running with account address:", address)
    if address != account.address:
        print("Running with agent address:", account.address)
    info = Info(base_url, skip_ws, perp_dexs=perp_dexs)
    user_state = info.user_state(address)
    spot_user_state = info.spot_user_state(address)
    margin_summary = user_state["marginSummary"]
    if float(margin_summary["accountValue"]) == 0 and len(spot_user_state["balances"]) == 0:
        print("Not running the example because the provided account has no equity.")
        url = info.base_url.split(".", 1)[1]
        error_string = f"No accountValue:\nIf you think this is a mistake, make sure that {address} has a balance on {url}.\nIf address shown is your API wallet address, update the config to specify the address of your account, not the address of the API wallet."
        raise Exception(error_string)
    exchange = Exchange(account, base_url, account_address=address, perp_dexs=perp_dexs)
    return address, info, exchange

In [ ]:
#| export
def retrieve_hyperliquid_perp_price(coin="ETH", interval="1h", 
                                end_date=datetime.now().strftime('%Y-%m-%dT%H:%M:%SZ'),
                                start_date=(datetime.now()-pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                info=None):
    """
    Retrieves historical candle data from Hyperliquid for a given coin and time interval.

    Args:
        coin (str, optional): Coin symbol (e.g. "ETH"). Defaults to "ETH".
        interval (str, optional): Candle interval ("1m", "5m", "15m", "1h", "4h", "1d"). Defaults to "1h".
        end_date (str, optional): End datetime in ISO 8601 format. Defaults to current UTC time.
        start_date (str, optional): Start datetime in ISO 8601 format. Defaults to 2 days before end_date.
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.

    Returns:
        pandas.DataFrame: DataFrame containing the OHLCV data with columns:
            - datetime: Timestamp for the candle (UTC)
            - open: Opening price of the interval
            - high: Highest traded price in the interval
            - low: Lowest traded price in the interval
            - close: Closing price of the interval
            - volume: Trading volume in the interval
            - coin: Coin symbol
        Returns None if the API request fails or returns no data.

    Notes:
        - All datetime values are in UTC timezone
        - Requires Hyperliquid Info client to be initialized
    """
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    try:
        # Convert datetime strings to Unix milliseconds timestamps
        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date)
        
        start_time_ms = int(start_dt.timestamp() * 1000)
        end_time_ms = int(end_dt.timestamp() * 1000)
        
        # Get candles from Hyperliquid
        candles = info.candles_snapshot(name=coin, interval=interval, 
                                       startTime=start_time_ms, endTime=end_time_ms)
        
        if not candles:
            return None
        
        # Convert to DataFrame
        df = pd.DataFrame(candles)
        
        # Convert timestamp to datetime
        df['datetime'] = pd.to_datetime(df['t'], unit='ms')
        
        # Rename columns to match coinbase format
        df = df.rename(columns={
            'o': 'open',
            'h': 'high', 
            'l': 'low',
            'c': 'close',
            'v': 'volume'
        })
        
        # Add coin column
        df['coin'] = coin
        
        # Sort by datetime and reorder columns
        df = df.sort_values(by='datetime')
        df = df[['datetime', 'open', 'high', 'low', 'close', 'volume', 'coin']]
        df = df.astype({'open': 'float64', 'high': 'float64', 'low': 'float64', 'close': 'float64', 'volume': 'float64'})
        
        return df.reset_index(drop=True)
        
    except Exception as e:
        print(f"Error retrieving candles for {coin}: {e}")
        return None

### Setup Hyperliquid Info client

You will need a api key and secret key from the Hyperliquid API. Store this into the `config_hyperliquid.json` file in the same directory as your script. See: https://app.hyperliquid.xyz/API

The best is to first call the `setup` function once to initialize the Hyperliquid Info client. Then pass "info" to any function that requires it. That will avoid calling the `setup` function multiple times.

In [ ]:
#| eval:false
address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)

Running with account address: 0x92c9F5BB0D5e6a7c09795324b6522bE9415F5fB1
Running with agent address: 0xeeDc9Da51290C624aB19E0ED33591793DEcC9C7C



### Example usage: Retrieve OHLCV data for Ethereum (ETH) at 1-hour intervals

Typical usage:

In [ ]:
#| eval: false
perp = retrieve_hyperliquid_perp_price(coin="ETH", interval="1h",info=info)
print(perp.head())

             datetime    open    high     low   close      volume coin
0 2025-10-11 17:00:00  3817.8  3839.1  3810.9  3823.2   9181.5874  ETH
1 2025-10-11 18:00:00  3823.2  3823.7  3809.5  3813.2   9780.7389  ETH
2 2025-10-11 19:00:00  3813.2  3813.4  3729.6  3758.0  43335.3442  ETH
3 2025-10-11 20:00:00  3758.1  3763.7  3659.2  3692.2  67099.0888  ETH
4 2025-10-11 21:00:00  3692.2  3742.1  3641.6  3733.8  40980.5741  ETH


In [ ]:
#| export
def spot_tickers(coin="ETH", base='USDC',info=None):
    """
    Retrieves current tickers for a given coin.

    Args:
        coin (str, optional): Coin symbol (e.g. "ETH"). Defaults to "ETH".
        base (str, optional): Coin symbol (e.g. "USDC"). Defaults to "USDC".
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.

    Returns:
        spot ticker (non intuitive symbol)
        Returns None if the API request fails or returns no data.

    Notes:
        - Requires Hyperliquid Info client to be initialized
    """
    if info is None:
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    maps =info.spot_meta_and_asset_ctxs()
    # change of ticker for ETH and BTC.... they should have U in front of the name.... Hyperliquid's naming convention...
    if coin.upper() == "ETH":
        coin = "UETH"
    elif coin.upper() == "BTC":
        coin = "UBTC"
    elif coin.upper() == "DOGE":
        coin = "UDOGE"
    elif coin.upper() == "SOL":
        coin = "USOL"
    if base.upper() == "USDC":
        base = "USDC"
    # Find the index of the coin and base in the universe of tokens
    # If not found, return None
    id_coin, id_base = None, None
    for token_ctx in maps[0]['tokens']:
        if token_ctx['name'] == coin:
            id_coin = token_ctx['index']
        elif token_ctx['name'] == base:
            id_base = token_ctx['index']
    if id_coin is None or id_base is None:
        return None
    for i in maps[0]['universe']:
        if i['tokens'] == [id_coin,id_base]:
            return i['name']
    id_coin, id_base = None, None
    for token_ctx in maps[0]['tokens']:
        if token_ctx['name'] == coin:
            id_coin = token_ctx['index']
        elif token_ctx['name'] == base:
            id_base = token_ctx['index']
    if id_coin is None or id_base is None:
        return None
    for i in maps['universe']:
        if i['tokens'] == [id_coin,id_base]:
            return i['name']
    return None                

In [ ]:
#| export
def retrieve_hyperliquid_spot_price(coin="ETH", base='USDC',interval="1h", 
                                end_date=datetime.now().strftime('%Y-%m-%dT%H:%M:%SZ'),
                                start_date=(datetime.now()-pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                info=None):
    # get the ticker for the given coin
    ticker = spot_tickers(coin=coin,base=base,info=info)
    if ticker is None:
        print(f"{coin} is not listed.")
        return pd.DataFrame()
    # get the price for the given ticker... same function used for perpetuals but
    # ticker price is different...
    try:
        price = retrieve_hyperliquid_perp_price(coin=ticker, interval=interval, 
                                end_date=end_date,
                                start_date=start_date,
                                info=info)
        price['coin'] = coin
        return price
    except Exception as e:
        print(f"Error retrieving price for {coin}: {e}")
        return None

## Example usage: Retrieve spot price for Ethereum (ETH) at 1-hour intervals

The ticker for Ethereum (ETH) is "UETH". There are other unusual choices but the function handles these choices by changing the ticker. 

In [ ]:
#| eval: false
spot=retrieve_hyperliquid_spot_price(info=info)
print(spot.head())

             datetime    open    high     low   close     volume coin
0 2025-10-11 17:00:00  3823.4  3846.0  3818.0  3829.3   674.6962  ETH
1 2025-10-11 18:00:00  3828.8  3829.1  3815.5  3819.5   171.9790  ETH
2 2025-10-11 19:00:00  3818.3  3818.3  3731.8  3763.2  1654.8737  ETH
3 2025-10-11 20:00:00  3763.1  3769.3  3667.9  3700.5  2286.3727  ETH
4 2025-10-11 21:00:00  3699.9  3747.2  3650.0  3739.2  1728.0468  ETH


If you want to find out which ticker is used for a given coin, you can use the `spot_tickers` function. For example, for Ethereum (ETH) quoted in USDC is:

In [ ]:
#| eval: false
stk = spot_tickers(info=info,coin="ETH",base="USDC")
print(f'spot ticker for ETH: ', stk)

spot ticker for ETH:  @151


## List all tokens in Hyperliquid

In [ ]:
#| export
def hyperliquid_tokens(info=None,rm_delisted=True):
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)

    # Get universe details
    tokens = info.meta_and_asset_ctxs()[0].get('universe')
    
    # Create DataFrame with token details
    df = pd.DataFrame(tokens)
    df['isDelisted'] = ~df['isDelisted'].isna()
    df['onlyIsolated'] = ~df['onlyIsolated'].isna()
    df = df.loc[(~df['isDelisted']) & (~df['onlyIsolated'])]
    return df

In [ ]:
#| eval: false
tokens = hyperliquid_tokens(info)
print(tokens)

     szDecimals  name  maxLeverage  marginTableId  isDelisted  onlyIsolated
0             5   BTC           40             56       False         False
1             4   ETH           25             55       False         False
2             2  ATOM            5              5       False         False
4             1  DYDX            5              5       False         False
5             2   SOL           20             54       False         False
..          ...   ...          ...            ...         ...           ...
210           0    0G            3              3       False         False
211           0  HEMI            3              3       False         False
212           0  APEX            3              3       False         False
213           0    2Z            3              3       False         False
214           2   ZEC            5              5       False         False

[180 rows x 6 columns]


In [ ]:
#| export
def funding_calc(rate,premium,max_rate=0.0005,min_rate=-0.0005):
    return premium+max(min(rate-premium, max_rate), min_rate)

In [ ]:
#| export
def retrieve_hyperliquid_funding_history(coin="ETH", 
                                        end_date=datetime.now().strftime('%Y-%m-%dT%H:%M:%SZ'),
                                        start_date=(datetime.now()-pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                        info=None,
                                        calc=False):
    """
    Retrieves funding rate history from Hyperliquid for a given coin and time period.

    Args:
        coin (str, optional): Coin symbol (e.g. "ETH"). Defaults to "ETH".
        end_date (str, optional): End datetime in ISO 8601 format. Defaults to current UTC time.
        start_date (str, optional): Start datetime in ISO 8601 format. Defaults to 7 days before end_date.
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.

    Returns:
        pandas.DataFrame: DataFrame containing the funding history with columns:
            - datetime: Timestamp for the funding rate (UTC)
            - funding_rate: The funding rate value
            - premium: The premium component
            - coin: Coin symbol
        Returns None if the API request fails or returns no data.

    Notes:
        - All datetime values are in UTC timezone
        - Funding rates are typically updated every hour
        - Requires Hyperliquid Info client to be initialized
    """
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    try:
        # Convert datetime strings to Unix milliseconds timestamps
        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date)
        
        start_time_ms = int(start_dt.timestamp() * 1000)
        end_time_ms = int(end_dt.timestamp() * 1000)
        
        # Get funding history from Hyperliquid
        funding_rates = info.funding_history(
            name=coin,
            startTime=start_time_ms,
            endTime=end_time_ms
        )
        
        if not funding_rates:
            return None
        
        # Convert to DataFrame
        df = pd.DataFrame(funding_rates)
        
        # Convert timestamp from milliseconds to datetime
        df['datetime'] = pd.to_datetime(df['time'], unit='ms')
        
        # Rename columns for clarity
        df = df.rename(columns={
            'fundingRate': 'funding_rate',
            'premium': 'premium'
        })
        
        # Convert multiple columns to float
        df[['funding_rate', 'premium']] = df[['funding_rate', 'premium']].astype(float)
        # Add coin column
        df['coin'] = coin
        
        # Sort by datetime and reorder columns
        df = df.sort_values(by='datetime')
        df = df[['datetime', 'funding_rate', 'premium', 'coin']]
        
        # Drop the original time column if it exists
        if 'time' in df.columns:
            df = df.drop(columns=['time'])
        
        df['fund_calc'] = df['funding_rate']
        if calc:
            vectorized_funding_calc = np.vectorize(funding_calc)
            df['fund_calc'] = vectorized_funding_calc(df['funding_rate'], df['premium'])
        
        return df.reset_index(drop=True)
        
    except Exception as e:
        print(f"Error retrieving funding history for {coin}: {e}")
        return None

## Example usage for funding rate retrieval

In [ ]:
#| eval: false
f_r = retrieve_hyperliquid_funding_history(info=info,calc=True)
print(f_r.tail())

                  datetime  funding_rate   premium coin  fund_calc
43 2025-10-13 13:00:00.067      0.000013 -0.000327  ETH   0.000012
44 2025-10-13 14:00:00.029      0.000013 -0.000272  ETH   0.000012
45 2025-10-13 15:00:00.052      0.000013 -0.000377  ETH   0.000012
46 2025-10-13 16:00:00.027      0.000013 -0.000351  ETH   0.000012
47 2025-10-13 17:00:00.008      0.000013 -0.000365  ETH   0.000012


In [ ]:
#| eval: false
import plotly.express as px

# If you want to plot both funding rate and premium together
fig = px.line(f_r.melt(id_vars=['datetime', 'coin'], 
                       value_vars=['funding_rate', 'premium','fund_calc'],
                       var_name='metric', value_name='rate'),
              x='datetime', y='rate', color='metric',
              title='Ethereum Funding Rate and Premium Over Time')
fig.show()



## Unified function for easy data retrieval

In [ ]:
#| export
def retrieve_hyperliquid_data(ticker="ETH", 
                              data_type="perp",
                              start_date=None,
                              end_date=None,
                              lookback=2,
                              interval="1h",
                              base="USDC",
                              round_to_hour=False,
                              info=None):
    """
    Unified function to retrieve funding rates, perpetual prices, or spot prices from Hyperliquid.
    
    Args:
        ticker (str, optional): Coin symbol (e.g. "ETH", "BTC"). Defaults to "ETH".
        data_type (str, optional): Type of data to retrieve - "funding", "perp", or "spot". Defaults to "perp".
        start_date (str, optional): Start date as string. Can be:
            - ISO format: "2024-01-15T10:30:00Z" or "2024-01-15T10:30:00"
            - Date only: "2024-01-15"
            - If None, calculated from lookback. Defaults to None.
        end_date (str, optional): End date as string (same formats as start_date).
            - If None, uses current UTC time. Defaults to None.
        lookback (int, optional): Number of days to look back from end_date if start_date is None. 
            Defaults to 2.
        interval (str, optional): Candle interval for perp/spot data ("1m", "5m", "15m", "1h", "4h", "1d"). 
            Defaults to "1h". Not used for funding rates.
        base (str, optional): Base currency for spot prices (e.g. "USDC"). Defaults to "USDC".
            Not used for perp or funding rates.
        round_to_hour (bool, optional): If True, rounds start_date and end_date to nearest hour.
            Useful for funding rates which update hourly. Defaults to False.
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
    
    Returns:
        pandas.DataFrame: DataFrame containing the requested data with appropriate columns:
            - For "funding": datetime, funding_rate, premium, coin
            - For "perp": datetime, open, high, low, close, volume, coin
            - For "spot": datetime, open, high, low, close, volume, coin
        Returns None if the API request fails or returns no data.
    
    Examples:
        # Get 7 days of funding rates for ETH, rounded to hour
        df = retrieve_hyperliquid_data("ETH", "funding", lookback=7, round_to_hour=True, info=info)
        
        # Get perp prices between specific dates with 4h interval
        df = retrieve_hyperliquid_data("BTC", "perp", 
                                      start_date="2024-01-01", 
                                      end_date="2024-01-15",
                                      interval="4h", info=info)
        
        # Get spot prices for last 30 days with 1h interval
        df = retrieve_hyperliquid_data("ETH", "spot", lookback=30, 
                                      interval="1h", base="USDC", info=info)
    
    Notes:
        - All datetime values are in UTC timezone
        - Valid data_type values: "funding", "perp", "spot"
        - Funding rates are updated hourly, so round_to_hour=True is recommended
        - Requires Hyperliquid Info client to be initialized
    """
    # Validate data_type
    valid_types = ["funding", "perp", "spot"]
    if data_type not in valid_types:
        raise ValueError(f"data_type must be one of {valid_types}, got '{data_type}'")
    
    # Initialize info client if not provided
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    # Handle end_date
    if end_date is None:
        end_dt = pd.Timestamp.now()
    else:
        # Parse end_date string
        try:
            end_dt = pd.to_datetime(end_date)
        except Exception as e:
            print(f"Error parsing end_date '{end_date}': {e}")
            return None

    # Handle start_date
    if start_date is None:
        # Calculate from lookback
        start_dt = end_dt - pd.Timedelta(days=lookback)
    else:
        # Parse start_date string
        try:
            start_dt = pd.to_datetime(start_date)
        except Exception as e:
            print(f"Error parsing start_date '{start_date}': {e}")
            return None
    
    # Convert to ISO format strings
    start_date_str = start_dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    end_date_str = end_dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    
    # Call appropriate function based on data_type
    try:
        if data_type == "funding":
            df = retrieve_hyperliquid_funding_history(
                coin=ticker,
                start_date=start_date_str,
                end_date=end_date_str,
                info=info
                )
            if round_to_hour:
                df['datetime'] = df['datetime'].apply(lambda x: x.round('h'))
            return df
        elif data_type == "perp":
            return retrieve_hyperliquid_perp_price(
                coin=ticker,
                interval=interval,
                start_date=start_date_str,
                end_date=end_date_str,
                info=info
            )
        elif data_type == "spot":
            return retrieve_hyperliquid_spot_price(
                coin=ticker,
                base=base,
                interval=interval,
                start_date=start_date_str,
                end_date=end_date_str,
                info=info
            )
    except Exception as e:
        print(f"Error retrieving {data_type} data for {ticker}: {e}")
        return None

In [ ]:
#| eval: false
## Example 1: Get funding rates for last 7 days, rounded to hour
funding_df = retrieve_hyperliquid_data(
    ticker="ETH",
    data_type="funding",
    lookback=7,
    round_to_hour=True,
    info=info
)
print(funding_df.head())

             datetime  funding_rate   premium coin  fund_calc
0 2025-10-06 18:00:00      0.000013  0.000475  ETH   0.000013
1 2025-10-06 19:00:00      0.000013  0.000389  ETH   0.000013
2 2025-10-06 20:00:00      0.000013  0.000346  ETH   0.000013
3 2025-10-06 21:00:00      0.000013  0.000318  ETH   0.000013
4 2025-10-06 22:00:00      0.000013  0.000293  ETH   0.000013


In [ ]:
#| eval: false
## Example 2: Get perpetual prices with specific dates and 4h interval
perp_df = retrieve_hyperliquid_data(
    ticker="BTC",
    data_type="perp",
    start_date="2024-01-01",
    end_date="2024-01-15",
    interval="4h",
    info=info
)
print(perp_df.head())

             datetime     open     high      low    close     volume coin
0 2024-01-01 00:00:00  42327.0  42820.0  42292.0  42361.0   55.76506  BTC
1 2024-01-01 04:00:00  42364.0  42551.0  42225.0  42535.0   22.11124  BTC
2 2024-01-01 08:00:00  42542.0  42805.0  42491.0  42727.0   47.98588  BTC
3 2024-01-01 12:00:00  42727.0  42911.0  42646.0  42847.0  171.11967  BTC
4 2024-01-01 16:00:00  42839.0  43616.0  42711.0  43587.0   92.64396  BTC


In [ ]:
#| eval: false
## Example 3: Get spot prices for last 30 days
spot_df = retrieve_hyperliquid_data(
    ticker="ETH",
    data_type="spot",
    lookback=30,
    interval="1h",
    base="USDC",
    info=info
)
print(spot_df.head())


             datetime    open    high     low   close    volume coin
0 2025-09-13 17:00:00  4625.6  4644.1  4608.7  4636.5  821.4446  ETH
1 2025-09-13 18:00:00  4636.4  4654.0  4627.7  4646.0  852.8406  ETH
2 2025-09-13 19:00:00  4646.1  4650.5  4638.3  4640.6  200.1443  ETH
3 2025-09-13 20:00:00  4640.5  4663.7  4639.6  4660.0  352.2797  ETH
4 2025-09-13 21:00:00  4660.0  4669.5  4659.5  4665.2  272.0407  ETH


In [ ]:
#| eval: false
## Example 4: Using datetime strings with time information
data_df = retrieve_hyperliquid_data(
    ticker="ETH",
    data_type="perp",
    start_date="2025-03-15T10:30:00",
    end_date="2025-03-20T15:45:00",
    interval="1h",
    info=info
)
print(data_df.head())

             datetime    open    high     low   close      volume coin
0 2025-03-19 11:00:00  1973.4  2033.0  1973.4  2027.9  44953.8887  ETH
1 2025-03-19 12:00:00  2027.8  2027.9  1995.0  2003.7  22597.9539  ETH
2 2025-03-19 13:00:00  2003.7  2022.3  1997.7  2016.4  28127.5482  ETH
3 2025-03-19 14:00:00  2016.5  2040.5  2016.2  2030.6  22843.6461  ETH
4 2025-03-19 15:00:00  2030.7  2056.7  2026.0  2048.7  30206.2723  ETH


In [ ]:
#| eval: false
## Example 5: Get funding rates with date-only strings
funding_df = retrieve_hyperliquid_data(
    ticker="BTC",
    data_type="funding",
    start_date="2025-08-29",
    end_date="2025-09-01",
    round_to_hour=True,
    info=info
)
print(funding_df)

              datetime  funding_rate   premium coin  fund_calc
0  2025-08-29 00:00:00      0.000013  0.000251  BTC   0.000013
1  2025-08-29 01:00:00      0.000013  0.000234  BTC   0.000013
2  2025-08-29 02:00:00      0.000013  0.000124  BTC   0.000013
3  2025-08-29 03:00:00      0.000013  0.000068  BTC   0.000013
4  2025-08-29 04:00:00      0.000013  0.000086  BTC   0.000013
..                 ...           ...       ...  ...        ...
67 2025-08-31 19:00:00      0.000013 -0.000035  BTC   0.000013
68 2025-08-31 20:00:00      0.000013 -0.000082  BTC   0.000013
69 2025-08-31 21:00:00      0.000013 -0.000028  BTC   0.000013
70 2025-08-31 22:00:00      0.000013 -0.000059  BTC   0.000013
71 2025-08-31 23:00:00      0.000013 -0.000012  BTC   0.000013

[72 rows x 5 columns]


In [ ]:
#| eval: false
data_df = retrieve_hyperliquid_data(
    ticker="ETH",
    data_type="perp",
    start_date="2025-03-15T10:30:00",
    end_date="2025-03-20T15:45:00",
    interval="1h",
    info=info
)
data_df.tail()

,datetime,open,high,low,close,volume,coin
24,2025-03-20 11:00:00,1981.4,1992.8,1981.4,1988.0,9136.9044,ETH
25,2025-03-20 12:00:00,1988.1,1997.7,1984.0,1995.8,12555.9900,ETH
26,2025-03-20 13:00:00,1995.8,1997.6,1977.7,1986.1,17570.3728,ETH
27,2025-03-20 14:00:00,1985.4,2010.0,1984.9,2002.4,31388.7768,ETH
28,2025-03-20 15:00:00,2002.4,2003.1,1965.5,1971.6,46490.5477,ETH


In [ ]:
data_df.head()

,datetime,open,high,low,close,volume,coin
0,2025-03-19 15:00:00,2030.7,2056.7,2026.0,2048.7,30206.2723,ETH
1,2025-03-19 16:00:00,2048.7,2049.2,2035.3,2047.4,22198.4803,ETH
2,2025-03-19 17:00:00,2047.4,2048.9,2014.3,2026.1,35909.8007,ETH
3,2025-03-19 18:00:00,2026.1,2059.9,1998.0,2045.0,84148.3569,ETH
4,2025-03-19 19:00:00,2045.1,2052.1,2020.1,2029.7,38921.5619,ETH


## Order book data

The will return the snapshot of the order book for the specified ticker in the moment you call the function. It does not provide historical data. 

This is good only for perpetual markets.

In [ ]:
def unix_to_datetime(t):
    return datetime.fromtimestamp(t/1000).strftime("%Y-%m-%d %H:%M:%S.%f")
#The funding rates are reset every hour.
#t = datetime.datetime.fromtimestamp(i['time']/1000).strftime("%Y-%m-%d %H:%M:%S.%f")
#print(t)

In [ ]:

#| export
def retrieve_hyperliquid_l2_snapshot(coin="ETH", info=None):
    """
    Retrieves current L2 order book snapshot from Hyperliquid for a given coin.
    
    Args:
        coin (str, optional): Coin symbol (e.g. "ETH", "BTC"). Defaults to "ETH".
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
    
    Returns:
        pandas.DataFrame: DataFrame containing the order book snapshot with columns:
            - datetime: Timestamp of the snapshot (UTC)
            - side: Order side ("bid" or "ask")
            - price: Price level
            - size: Total size at this price level
            - num_orders: Number of orders at this price level
        Returns None if the API request fails or returns no data.
    
    Notes:
        - This is a snapshot at the moment the function is called
        - All datetime values are in UTC timezone
        - Bids are sorted from highest to lowest price
        - Asks are sorted from lowest to highest price
        - Requires Hyperliquid Info client to be initialized
    """
    if info is None:
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    try:
        # Get L2 snapshot from Hyperliquid
        l2_data = info.l2_snapshot(name=coin)
        
        if not l2_data or 'levels' not in l2_data:
            return None
        
        # Convert timestamp to datetime
        timestamp = pd.to_datetime(l2_data['time'], unit='ms')
        
        # Extract bid and ask levels
        bids = l2_data['levels'][0]  # First element is bids
        asks = l2_data['levels'][1]  # Second element is asks
        
        # Create list to store all rows
        rows = []
        
        # Process bids
        for bid in bids:
            rows.append({
                'datetime': timestamp,
                'side': 'bid',
                'price': float(bid['px']),
                'size': float(bid['sz']),
                'num_orders': int(bid['n'])
            })
        
        # Process asks
        for ask in asks:
            rows.append({
                'datetime': timestamp,
                'side': 'ask',
                'price': float(ask['px']),
                'size': float(ask['sz']),
                'num_orders': int(ask['n'])
            })
        
        # Create DataFrame
        df = pd.DataFrame(rows)
        
        # Reorder columns for consistency
        df = df[['datetime', 'side', 'price', 'size', 'num_orders']]
        
        return df
        
    except Exception as e:
        print(f"Error retrieving L2 snapshot for {coin}: {e}")
        return None

In [ ]:
#| eval: false

# Example usage: Get L2 order book snapshot for ETH
l2_snapshot = retrieve_hyperliquid_l2_snapshot(coin="ETH", info=info)
print(l2_snapshot.head(5))
print(l2_snapshot.tail(5))

# Check the structure
print(f"\nTotal levels: {len(l2_snapshot)}")
print(f"Bids: {len(l2_snapshot[l2_snapshot['side'] == 'bid'])}")
print(f"Asks: {len(l2_snapshot[l2_snapshot['side'] == 'ask'])}")
print(f"Snapshot time: {l2_snapshot['datetime'].iloc[0]}")

                 datetime side   price      size  num_orders
0 2025-10-13 22:16:51.383  bid  4259.2  273.4566          44
1 2025-10-13 22:16:51.383  bid  4259.1   42.6607           4
2 2025-10-13 22:16:51.383  bid  4259.0    3.5219           1
3 2025-10-13 22:16:51.383  bid  4258.9    4.3848           4
4 2025-10-13 22:16:51.383  bid  4258.8   64.5277           3
                  datetime side   price      size  num_orders
35 2025-10-13 22:16:51.383  ask  4260.8   20.7258          18
36 2025-10-13 22:16:51.383  ask  4260.9  268.0489          18
37 2025-10-13 22:16:51.383  ask  4261.0  228.3555          13
38 2025-10-13 22:16:51.383  ask  4261.1  208.6194           8
39 2025-10-13 22:16:51.383  ask  4261.2   64.1188          18

Total levels: 40
Bids: 20
Asks: 20
Snapshot time: 2025-10-13 22:16:51.383000


## Save data and build keep the history

Similar to coinbase.py, this script saves the data in a specified format. These functions handles duplicates and sorts by date. For hourly data, it aligns to hour

In [ ]:

#| export
def save_hyperliquid_file(df, folder_path, file_name, type="parquet"):
    """
    Save a pandas DataFrame to a file in either CSV or Parquet format.

    Args:
        df (pandas.DataFrame): The DataFrame to save
        folder_path (str): Directory path where the file will be saved
        file_name (str): Name of the file without extension
        type (str, optional): File format - either "csv" or "parquet". Defaults to "parquet"

    The function saves the DataFrame to the specified path, handling the file extension automatically.
    For CSV files, the index is not saved. For Parquet files, default Parquet settings are used.
    Creates the folder if it doesn't exist.
    """
    # Create folder if it doesn't exist
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    
    if type == "csv":
        df.to_csv(f"{folder_path}/{file_name}.csv", index=False)
    elif type == "parquet":
        df.to_parquet(f"{folder_path}/{file_name}.parquet")
    else:
        raise ValueError(f"Type {type} not supported. Use 'csv' or 'parquet'")

In [ ]:

#| export
def hyperliquid_perp_tokens(info=None):
    """
    Get list of available perpetual tokens on Hyperliquid.
    
    Args:
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
    
    Returns:
        list: List of perpetual token symbols (e.g., ['BTC', 'ETH', 'SOL', ...])
    """
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    tokens_df = hyperliquid_tokens(info)

    # Filter out delisted or inactive tokens if status column exists
    if 'name' in tokens_df.columns:
        return tokens_df['name'].tolist()
    
    return []

In [ ]:

#| export
def hyperliquid_spot_tokens(base="USDC", info=None):
    """
    Get list of available spot trading pairs on Hyperliquid.
    
    Args:
        base (str, optional): Base currency to filter by. Defaults to "USDC".
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
    
    Returns:
        pandas.DataFrame: DataFrame with columns:
            - coin: The cryptocurrency symbol
            - base: The base currency
            - ticker: The Hyperliquid ticker symbol
    """
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    # Get spot metadata
    maps = info.spot_meta_and_asset_ctxs()
    
    # Build list of available pairs
    pairs = []
    
    # Get all tokens
    tokens = {token['index']: token['name'] for token in maps[0]['tokens']}
    
    # Find base currency index
    base_index = None
    for idx, name in tokens.items():
        if name == base:
            base_index = idx
            break
    
    if base_index is None:
        print(f"Base currency {base} not found")
        return pd.DataFrame(columns=['coin', 'base', 'ticker'])
    
    # Get all trading pairs with this base
    for universe_item in maps[0]['universe']:
        token_indices = universe_item['tokens']
        if len(token_indices) == 2 and base_index in token_indices:
            # Find the other token (the coin)
            coin_index = token_indices[0] if token_indices[1] == base_index else token_indices[1]
            coin_name = tokens.get(coin_index, '')
            
            # Map back from Hyperliquid naming (UETH -> ETH, etc.)
            display_coin = coin_name
            if coin_name == "UETH":
                display_coin = "ETH"
            elif coin_name == "UBTC":
                display_coin = "BTC"
            elif coin_name == "UDOGE":
                display_coin = "DOGE"
            elif coin_name == "USOL":
                display_coin = "SOL"
            
            pairs.append({
                'coin': display_coin,
                'base': base,
                'ticker': universe_item['name']
            })
    
    return pd.DataFrame(pairs)

In [ ]:

#| export
def hyperliquid_perp_to_file(coins=None, folder_path="../data/hyperliquid/perp", 
                             interval="1h", start_date=None, refresh_24h=False, 
                             type="parquet", info=None, verbose=True):
    """
    Downloads and maintains historical perpetual price data for Hyperliquid tokens.
    
    Args:
        coins (list, optional): List of coin symbols to process. If None, processes all available perp tokens.
        folder_path (str): Path where token data files will be stored. Defaults to "../data/hyperliquid/perp"
        interval (str): Time interval between price points ("1m", "5m", "15m", "1h", "4h", "1d"). Defaults to "1h"
        start_date (str, optional): Start date for initial download if file doesn't exist. Format: 'YYYY-MM-DD'
        refresh_24h (bool): If True, replaces last 24 hours of data. Defaults to False
        type (str): File format - "csv" or "parquet". Defaults to "parquet"
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
        verbose (bool): If True, prints progress messages. Defaults to True
    
    The function:
    - Creates folder structure if it doesn't exist
    - For each coin, checks if data file exists:
        - If exists: Loads file and appends new data since last recorded date
        - If not exists: Downloads history from start_date (or last 30 days if not specified)
    - Saves data in specified format, handling duplicates and sorting by datetime
    - All datetime values are in UTC timezone
    """
    # Initialize info client if needed
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    # Get list of coins to process
    if coins is None:
        coins = hyperliquid_perp_tokens(info)
        if verbose:
            print(f"Found {len(coins)} perpetual tokens")
    
    # Create folder if it doesn't exist
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        if verbose:
            print(f"Created folder: {folder_path}")
    
    # Track successes and failures
    success_count = 0
    failure_count = 0
    failed_coins = []
    
    # Process each coin
    for coin in coins:
        if verbose:
            print(f"Processing {coin}...")
        
        file_name = f"{folder_path}/{coin}.{type}"
        
        try:
            # Check if file exists
            if os.path.exists(file_name):
                # Load existing data
                if type == "csv":
                    df = pd.read_csv(file_name)
                elif type == "parquet":
                    df = pd.read_parquet(file_name)
                else:
                    raise ValueError(f"Type {type} not supported")
                
                # Get last date in file
                df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
                last_date = df['datetime'].iloc[-1]
                today = pd.Timestamp.now(tz='UTC')
                
                # Handle refresh_24h option
                first_date = today - pd.Timedelta(hours=24)
                if first_date < last_date and refresh_24h:
                    df = df[df['datetime'] < first_date]
                    last_date = first_date
                    if verbose:
                        print(f"  Refreshing last 24 hours for {coin}")
                
                # Check if update is needed
                if last_date < today:
                    # Download new data
                    df_new = retrieve_hyperliquid_perp_price(
                        coin=coin,
                        interval=interval,
                        start_date=last_date.strftime('%Y-%m-%dT%H:%M:%SZ'),
                        end_date=today.strftime('%Y-%m-%dT%H:%M:%SZ'),
                        info=info
                    )
                    
                    if df_new is not None and not df_new.empty:
                        # Combine and clean data
                        df = pd.concat([df, df_new])
                        df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
                        df = df.drop_duplicates(subset='datetime').sort_values(by='datetime').reset_index(drop=True)
                        
                        # Save updated data
                        save_hyperliquid_file(df, folder_path, coin, type)
                        if verbose:
                            print(f"  Updated {coin}: added {len(df_new)} new records")
                        success_count += 1
                    else:
                        if verbose:
                            print(f"  {coin} is up to date")
                        success_count += 1
                else:
                    if verbose:
                        print(f"  {coin} is up to date")
                    success_count += 1
            
            else:
                # File doesn't exist - download full history
                if start_date is None:
                    # Default to 30 days back
                    start_date_str = (pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=300)).strftime('%Y-%m-%dT%H:%M:%SZ')
                else:
                    start_date_str = pd.to_datetime(start_date).strftime('%Y-%m-%dT%H:%M:%SZ')
                
                end_date_str = pd.Timestamp.now(tz='UTC').strftime('%Y-%m-%dT%H:%M:%SZ')
                
                df = retrieve_hyperliquid_perp_price(
                    coin=coin,
                    interval=interval,
                    start_date=start_date_str,
                    end_date=end_date_str,
                    info=info
                )
                
                if df is not None and not df.empty:
                    save_hyperliquid_file(df, folder_path, coin, type)
                    if verbose:
                        print(f"  Downloaded {coin}: {len(df)} records")
                    success_count += 1
                else:
                    if verbose:
                        print(f"  Failed to download {coin}")
                    failure_count += 1
                    failed_coins.append(coin)
        
        except Exception as e:
            if verbose:
                print(f"Error processing {coin}: {e}")

In [ ]:

#| eval: false
import tempfile
import shutil

# Create a test DataFrame
test_df = pd.DataFrame({
    'datetime': pd.date_range('2024-01-01', periods=5, freq='1h'),
    'open': [100.0, 101.0, 102.0, 103.0, 104.0],
    'high': [105.0, 106.0, 107.0, 108.0, 109.0],
    'low': [95.0, 96.0, 97.0, 98.0, 99.0],
    'close': [102.0, 103.0, 104.0, 105.0, 106.0],
    'volume': [1000.0, 1100.0, 1200.0, 1300.0, 1400.0],
    'coin': ['ETH'] * 5
})

# Create temporary directory for testing
test_dir = tempfile.mkdtemp()

try:
    # Test 1: Save as CSV
    print("Test 1: Saving as CSV...")
    save_hyperliquid_file(test_df, test_dir, "test_eth", type="csv")
    csv_path = f"{test_dir}/test_eth.csv"
    assert os.path.exists(csv_path), "CSV file was not created"
    loaded_csv = pd.read_csv(csv_path)
    assert len(loaded_csv) == len(test_df), "CSV data length mismatch"
    print("✓ CSV save test passed")
    
    # Test 2: Save as Parquet
    print("\nTest 2: Saving as Parquet...")
    save_hyperliquid_file(test_df, test_dir, "test_eth", type="parquet")
    parquet_path = f"{test_dir}/test_eth.parquet"
    assert os.path.exists(parquet_path), "Parquet file was not created"
    loaded_parquet = pd.read_parquet(parquet_path)
    assert len(loaded_parquet) == len(test_df), "Parquet data length mismatch"
    print("✓ Parquet save test passed")
    
    # Test 3: Invalid file type
    print("\nTest 3: Testing invalid file type...")
    try:
        save_hyperliquid_file(test_df, test_dir, "test_eth", type="invalid")
        print("✗ Should have raised ValueError for invalid type")
    except ValueError as e:
        print(f"✓ Correctly raised ValueError: {e}")
    
    # Test 4: Create new directory
    print("\nTest 4: Testing directory creation...")
    new_dir = f"{test_dir}/new_folder"
    save_hyperliquid_file(test_df, new_dir, "test_eth", type="csv")
    assert os.path.exists(new_dir), "New directory was not created"
    assert os.path.exists(f"{new_dir}/test_eth.csv"), "File was not created in new directory"
    print("✓ Directory creation test passed")
    
finally:
    # Cleanup
    shutil.rmtree(test_dir)
    print("\n✓ All save_hyperliquid_file tests completed")

Test 1: Saving as CSV...
✓ CSV save test passed

Test 2: Saving as Parquet...
✓ Parquet save test passed

Test 3: Testing invalid file type...
✓ Correctly raised ValueError: Type invalid not supported. Use 'csv' or 'parquet'

Test 4: Testing directory creation...
✓ Directory creation test passed

✓ All save_hyperliquid_file tests completed


In [ ]:

#| eval: false
print("Test: Getting perpetual tokens list...")
try:
    # Initialize info client
    #address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    # Get perp tokens
    perp_tokens = hyperliquid_perp_tokens(info=info)
    
    # Verify results
    assert isinstance(perp_tokens, list), "Result should be a list"
    assert len(perp_tokens) > 0, "Should return at least some tokens"
    
    # Check for common tokens
    common_tokens = ['BTC', 'ETH', 'SOL']
    found_tokens = [token for token in common_tokens if token in perp_tokens]
    assert len(found_tokens) > 0, f"Should find at least one common token from {common_tokens}"
    
    print(f"✓ Found {len(perp_tokens)} perpetual tokens")
    print(f"✓ Sample tokens: {perp_tokens[:10]}")
    print("✓ hyperliquid_perp_tokens test passed")
    
except Exception as e:
    print(f"✗ Test failed with error: {e}")

Test: Getting perpetual tokens list...
✓ Found 180 perpetual tokens
✓ Sample tokens: ['BTC', 'ETH', 'ATOM', 'DYDX', 'SOL', 'AVAX', 'BNB', 'APE', 'OP', 'LTC']
✓ hyperliquid_perp_tokens test passed


In [ ]:
#| eval: false
hyperliquid_perp_to_file(interval="1h",type="parquet",info=info,verbose=True)

Found 180 perpetual tokens
Created folder: ../data/hyperliquid/perp
Processing BTC...
  Downloaded BTC: 5002 records
Processing ETH...
  Downloaded ETH: 5002 records
Processing ATOM...
  Downloaded ATOM: 5001 records
Processing DYDX...
  Downloaded DYDX: 5001 records
Processing SOL...
  Downloaded SOL: 5001 records
Processing AVAX...
  Downloaded AVAX: 5001 records
Processing BNB...
  Downloaded BNB: 5001 records
Processing APE...
  Downloaded APE: 5001 records
Processing OP...
  Downloaded OP: 5001 records
Processing LTC...
  Downloaded LTC: 5001 records
Processing ARB...
  Downloaded ARB: 5001 records
Processing DOGE...
  Downloaded DOGE: 5001 records
Processing INJ...
  Downloaded INJ: 5001 records
Processing SUI...
  Downloaded SUI: 5001 records
Processing kPEPE...
  Downloaded kPEPE: 5001 records
Processing CRV...
  Downloaded CRV: 5001 records
Processing LDO...
  Downloaded LDO: 5001 records
Processing LINK...
  Downloaded LINK: 5001 records
Processing STX...
  Downloaded STX: 50

In [ ]:
#| eval: false
df=pd.read_parquet("../data/hyperliquid/perp/ETH.parquet")
df

,datetime,open,high,low,close,volume,coin
0,2025-03-19 15:00:00,2030.7,2056.7,2026.0,2048.7,30206.2723,ETH
1,2025-03-19 16:00:00,2048.7,2049.2,2035.3,2047.4,22198.4803,ETH
2,2025-03-19 17:00:00,2047.4,2048.9,2014.3,2026.1,35909.8007,ETH
3,2025-03-19 18:00:00,2026.1,2059.9,1998.0,2045.0,84148.3569,ETH
4,2025-03-19 19:00:00,2045.1,2052.1,2020.1,2029.7,38921.5619,ETH
...,...,...,...,...,...,...,...
4997,2025-10-13 20:00:00,4249.6,4287.3,4241.0,4284.4,36601.0015,ETH
4998,2025-10-13 21:00:00,4284.4,4288.4,4253.0,4256.6,20308.4485,ETH
4999,2025-10-13 22:00:00,4256.6,4274.7,4250.1,4269.6,12708.1801,ETH
5000,2025-10-13 23:00:00,4269.6,4269.6,4234.3,4238.9,10955.4105,ETH


In [ ]:
#| export
def hyper_funding_to_file(ticker=None,
                         start_date=None,
                         end_date=None,
                         lookback=2,
                         round_to_hour=True,
                         info=None,
                         data_dir="../data/hyperliquid/funding",
                         type="parquet",
                         update_mode=False,
                         refresh_24h=False):
    """
    Retrieves funding rate data from Hyperliquid and saves it to file(s).
    
    Args:
        ticker (str or None, optional): Coin symbol (e.g. "ETH", "BTC"). 
            If None, processes all available tokens from hyperliquid_tokens(). Defaults to None.
        start_date (str, optional): Start date as string. Can be:
            - ISO format: "2024-01-15T10:30:00Z" or "2024-01-15T10:30:00"
            - Date only: "2024-01-15"
            - If None, calculated from lookback. Defaults to None.
        end_date (str, optional): End date as string (same formats as start_date).
            - If None, uses current UTC time. Defaults to None.
        lookback (int, optional): Number of days to look back from end_date if start_date is None. 
            Defaults to 2.
        round_to_hour (bool, optional): If True, rounds datetime to nearest hour.
            Recommended for funding rates which update hourly. Defaults to True.
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
        data_dir (str, optional): Directory to save the file. Defaults to "../data/hyperliquid/funding".
        type (str, optional): File format - "parquet" or "csv". Defaults to "parquet".
        update_mode (bool, optional): If True, appends new data to existing file and removes duplicates.
            If False, overwrites existing file. Defaults to False.
        refresh_24h (bool, optional): If True, removes and re-fetches the last 24 hours of data.
            Useful for ensuring data quality and getting latest updates. Defaults to False.
    
    Returns:
        pandas.DataFrame or dict: 
            - If ticker is specified: DataFrame containing the funding rate data that was saved.
            - If ticker is None: Dictionary with ticker symbols as keys and DataFrames as values.
            Returns None if retrieval or save fails.
    
    Examples:
        # Save 7 days of funding rates for ETH
        df = hyper_funding_to_file("ETH", lookback=7, info=info)
        
        # Save funding data for all available tokens
        all_data = hyper_funding_to_file(lookback=7, info=info)
        
        # Update existing file with new data
        df = hyper_funding_to_file("BTC", lookback=1, update_mode=True, info=info)
        
        # Refresh last 24 hours and add new data for all tokens
        all_data = hyper_funding_to_file(update_mode=True, refresh_24h=True, info=info)
        
        # Save funding rates between specific dates
        df = hyper_funding_to_file("ETH", 
                                   start_date="2024-01-01", 
                                   end_date="2024-01-15",
                                   info=info)
    
    Notes:
        - All datetime values are in UTC timezone
        - In update_mode, duplicates are removed based on datetime column
        - File is saved as {ticker}.parquet or {ticker}.csv in data_dir
        - Creates data_dir if it doesn't exist
        - refresh_24h only works when update_mode=True and file exists
        - When ticker=None, processes all tokens and may take significant time
    """
    # Initialize info client if not provided
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    # If ticker is None, get all available tokens
    if ticker is None:
        tokens_df = hyperliquid_tokens(info=info)
        if tokens_df is None or tokens_df.empty:
            print("No tokens found")
            return None
        
        ticker_list = tokens_df['name'].tolist()
        print(f"Processing {len(ticker_list)} tokens: {', '.join(ticker_list[:10])}{'...' if len(ticker_list) > 10 else ''}")
        
        results = {}
        for i, tick in enumerate(ticker_list, 1):
            print(f"\n[{i}/{len(ticker_list)}] Processing {tick}...")
            try:
                df = hyper_funding_to_file(
                    ticker=tick,
                    start_date=start_date,
                    end_date=end_date,
                    lookback=lookback,
                    round_to_hour=round_to_hour,
                    info=info,
                    data_dir=data_dir,
                    type=type,
                    update_mode=update_mode,
                    refresh_24h=refresh_24h
                )
                if df is not None:
                    results[tick] = df
            except Exception as e:
                print(f"Error processing {tick}: {e}")
                continue
        
        print(f"\nCompleted: {len(results)}/{len(ticker_list)} tokens processed successfully")
        return results
    
    # Single ticker processing
    try:
        # Handle update mode with existing file
        existing_df = None
        if update_mode:
            file_path = os.path.join(data_dir, f"{ticker}.{type}")
            if os.path.exists(file_path):
                # Load existing data
                if type == "parquet":
                    existing_df = pd.read_parquet(file_path)
                elif type == "csv":
                    existing_df = pd.read_csv(file_path)
                    existing_df['datetime'] = pd.to_datetime(existing_df['datetime'])
                else:
                    raise ValueError(f"Unsupported file type: {type}")
                
                # Handle refresh_24h: remove last 24 hours of data
                if refresh_24h:
                    import datetime as dt
                    cutoff_time = datetime.now(tz=dt.UTC) - dt.timedelta(hours=24)
                    existing_df['datetime'] = pd.to_datetime(existing_df['datetime'], utc=True)
                    rows_before = len(existing_df)
                    existing_df = existing_df[existing_df['datetime'] < cutoff_time]
                    rows_removed = rows_before - len(existing_df)
                    print(f"Removed {rows_removed} rows from last 24 hours for refresh")
                    
                    # Adjust start_date to fetch from 24 hours ago
                    if start_date is None:
                        start_date = cutoff_time.strftime('%Y-%m-%dT%H:%M:%SZ')
                        lookback = None  # Override lookback when using refresh_24h
                
                # Get the last date in existing data to fetch new data from there
                if not existing_df.empty and start_date is None and not refresh_24h:
                    last_date = pd.to_datetime(existing_df['datetime'].max())
                    if pd.notna(last_date):
                        start_date = last_date.strftime('%Y-%m-%dT%H:%M:%SZ')
                        lookback = None  # Override lookback when we have existing data
        
        # Retrieve funding rate data
        df = retrieve_hyperliquid_data(
            ticker=ticker,
            data_type="funding",
            start_date=start_date,
            end_date=end_date,
            lookback=lookback if lookback is not None else 2,
            round_to_hour=round_to_hour,
            info=info
        )
        
        if df is None or df.empty:
            print(f"No funding rate data retrieved for {ticker}")
            if existing_df is not None and not existing_df.empty:
                print(f"Keeping existing data with {len(existing_df)} rows")
                save_hyperliquid_file(existing_df, data_dir, ticker, type=type)
                return existing_df
            return None
        
        # Combine with existing data if in update mode
        if update_mode and existing_df is not None and not existing_df.empty:
            df = pd.concat([existing_df, df], ignore_index=True)
            df = df.drop_duplicates(subset=['datetime'], keep='last')
            df = df.sort_values('datetime').reset_index(drop=True)
            print(f"Updated {ticker} funding data: {len(existing_df)} -> {len(df)} rows")
        
        # Save to file
        save_hyperliquid_file(df, data_dir, ticker, type=type)
        print(f"Saved {len(df)} rows of funding rate data for {ticker}")
        
        return df
        
    except Exception as e:
        print(f"Error saving funding rate data for {ticker}: {e}")
        return None

In [ ]:
#| eval:false
a = hyper_funding_to_file(lookback=10, info=info,update_mode=True) #TODO: fix.... not working as expected

Processing 180 tokens: BTC, ETH, ATOM, DYDX, SOL, AVAX, BNB, APE, OP, LTC...

[1/180] Processing BTC...
Updated BTC funding data: 240 -> 240 rows
Saved 240 rows of funding rate data for BTC

[2/180] Processing ETH...
Updated ETH funding data: 240 -> 240 rows
Saved 240 rows of funding rate data for ETH

[3/180] Processing ATOM...
Updated ATOM funding data: 240 -> 240 rows
Saved 240 rows of funding rate data for ATOM

[4/180] Processing DYDX...
Updated DYDX funding data: 240 -> 240 rows
Saved 240 rows of funding rate data for DYDX

[5/180] Processing SOL...
Updated SOL funding data: 240 -> 240 rows
Saved 240 rows of funding rate data for SOL

[6/180] Processing AVAX...
Updated AVAX funding data: 240 -> 240 rows
Saved 240 rows of funding rate data for AVAX

[7/180] Processing BNB...
Updated BNB funding data: 240 -> 240 rows
Saved 240 rows of funding rate data for BNB

[8/180] Processing APE...
Updated APE funding data: 240 -> 240 rows
Saved 240 rows of funding rate data for APE

[9/180] P

In [ ]:
retrieve_hyperliquid_data(
            ticker='ETH',
            data_type="spot",
            lookback=30,
            info=info
        )

,datetime,open,high,low,close,volume,coin
0,2025-09-13 20:00:00,4640.5,4663.7,4639.6,4660.0,352.2797,ETH
1,2025-09-13 21:00:00,4660.0,4669.5,4659.5,4665.2,272.0407,ETH
2,2025-09-13 22:00:00,4665.9,4668.0,4651.0,4661.7,190.1211,ETH
3,2025-09-13 23:00:00,4661.6,4674.9,4659.5,4669.3,281.7104,ETH
4,2025-09-14 00:00:00,4669.6,4684.1,4662.5,4674.0,380.6305,ETH
...,...,...,...,...,...,...,...
716,2025-10-13 16:00:00,4139.4,4175.4,4134.0,4167.8,1214.7652,ETH
717,2025-10-13 17:00:00,4167.9,4239.8,4161.6,4237.0,1950.5279,ETH
718,2025-10-13 18:00:00,4240.0,4240.0,4205.1,4232.7,1191.0993,ETH
719,2025-10-13 19:00:00,4232.7,4274.4,4229.1,4255.9,2022.5884,ETH


## More examples

Load package:

In [ ]:
from token_data.hyperliquid import *